# 10 - 基础数据采集管线

一键完成全部数据增量采集，并汇总展示数据概览。

**运行逻辑：**
- 自动检测缺失数据，只拉取增量部分，重复运行无额外开销
- 采集完成后展示：管线执行结果 → 全库数据总览 → 各标的明细 → 数据质量校验

**信号分析请使用** `11_signal_analysis.ipynb`

---

## 1. 初始化：数据库连接 + 建表

In [22]:
import time as _time
_t0 = _time.time()

from invest_model.db import get_engine
from invest_model.models.ddl import create_all_tables, ALL_TABLES

engine = get_engine()
created = create_all_tables(engine)
print(f"✅ {len(created)} 张表就绪: {', '.join(ALL_TABLES)}")

✅ 20 张表就绪: trade_calendar, stock_info, stock_pool, stock_daily, stock_fundamental, stock_fina_indicator, stock_cashflow, stock_holder_trade, stock_holder_count, stock_margin, index_daily, etf_info, etf_daily, stock_technical, etf_holding, stock_signal_snapshot, stock_composite_score, backtest_run, backtest_nav, backtest_trades


## 2. 运行增量管线

调用 `DailyPipeline.run("full")`，内部每个采集步骤自动检测缺失交易日，只拉取增量数据。

In [23]:
import pandas as pd
from IPython.display import display, HTML
from invest_model.pipeline.daily_pipeline import DailyPipeline

pipeline = DailyPipeline()
results = pipeline.run("full")

# 将管线结果转为表格展示
rows = []
for step, status in results.items():
    ok = status.startswith("OK")
    rows.append({"步骤": step, "状态": "✅ " + status if ok else "❌ " + status})

df_results = pd.DataFrame(rows)
print(f"\n{'='*50}")
print(f"管线执行完毕，共 {len(results)} 个步骤")
print(f"成功: {sum(1 for s in results.values() if s.startswith('OK'))}, "
      f"失败: {sum(1 for s in results.values() if s.startswith('FAILED'))}")
print(f"{'='*50}\n")
display(df_results)

18:36:48 | INFO    | Tushare 使用自定义接口基地址: http://118.89.66.41:8010/
18:36:48 | INFO    | Tushare 客户端初始化完成（Token 验证通过）
18:36:48 | INFO    | ============================================================
18:36:48 | INFO    | 每日管线启动 [full]: 2026-05-02 18:36:48
18:36:48 | INFO    | 执行步骤: ['calendar', 'stock_list', 'stock_daily', 'etf_daily', 'index_daily', 'daily_basic', 'technical', 'margin', 'cashflow', 'holder_trade', 'holder_count', 'signal_generation', 'validation']
18:36:48 | INFO    | ============================================================
18:36:48 | INFO    | --- [交易日历] 开始 ---
18:36:48 | INFO    | 交易日历已覆盖到 20261231，无需更新
18:36:48 | INFO    | --- [交易日历] 完成: 0 ---
18:36:48 | INFO    | --- [股票列表] 开始 ---
18:36:48 | INFO    | 开始采集 A 股股票列表...
18:36:49 | INFO    | 获取到 5512 只 A 股
18:36:50 | INFO    | 写入 stock_info: 5512 条
18:36:50 | INFO    | 开始采集 ETF 列表...
18:36:50 | INFO    | 获取到 1941 只 ETF
18:36:51 | INFO    | 写入 etf_info: 1941 条
18:36:51 | INFO    | --- [股票列表] 完成: stocks=5512, etfs=19


管线执行完毕，共 13 个步骤
成功: 13, 失败: 0



,步骤,状态
0,calendar,✅ OK (0)
1,stock_list,"✅ OK (stocks=5512, etfs=1941)"
2,stock_daily,"✅ OK (10 只, 新增 10 条)"
3,etf_daily,"✅ OK (1 只, 新增 1 条)"
4,index_daily,"✅ OK (5 指数, 新增 5 条)"
5,daily_basic,✅ OK (5460)
6,technical,"✅ OK (10 只, 新增 10 条)"
7,margin,✅ OK (1)
8,cashflow,"✅ OK (5 只, 新增 5 条)"
9,holder_trade,✅ OK (6)


## 3. 全库数据总览

各表行数一览，快速确认数据是否完整。

In [24]:
from invest_model.repositories.base import BaseRepository

base_repo = BaseRepository(engine)

table_stats = []
for tbl in ALL_TABLES:
    try:
        if not base_repo.table_exists(tbl):
            table_stats.append({"表名": tbl, "行数": "—", "状态": "❌ 不存在"})
            continue
        cnt = base_repo.get_row_count(tbl)
        table_stats.append({"表名": tbl, "行数": f"{cnt:,}", "状态": "✅" if cnt > 0 else "⚠️ 空表"})
    except Exception as e:
        table_stats.append({"表名": tbl, "行数": "—", "状态": f"❌ {e}"})

df_overview = pd.DataFrame(table_stats)
total_rows = sum(int(r["行数"].replace(",", "")) for r in table_stats if r["行数"] not in ("—",))
print(f"全库共 {len(ALL_TABLES)} 张表，总计 {total_rows:,} 行数据\n")
display(df_overview)

全库共 20 张表，总计 6,238,057 行数据



,表名,行数,状态
0,trade_calendar,"2,191",✅
1,stock_info,"5,515",✅
2,stock_pool,6,✅
3,stock_daily,"12,254",✅
4,stock_fundamental,"6,065,976",✅
5,stock_fina_indicator,98,✅
6,stock_cashflow,125,✅
7,stock_holder_trade,220,✅
8,stock_holder_count,208,✅
9,stock_margin,166,✅


## 4. 行情表时间范围

各行情表的数据起止日期与记录数。

In [25]:
time_tables = [
    "stock_daily", "stock_technical", "stock_fundamental",
    "stock_cashflow", "stock_margin", "stock_holder_count",
    "index_daily", "etf_daily",
]

rows = []
for tbl in time_tables:
    try:
        if not base_repo.table_exists(tbl):
            rows.append({"表": tbl, "最早日期": "—", "最新日期": "—", "记录数": 0, "标的数": 0})
            continue
        df_range = base_repo.read_sql(
            f"SELECT MIN(trade_date) as min_d, MAX(trade_date) as max_d, "
            f"COUNT(*) as cnt, COUNT(DISTINCT code) as codes FROM `{tbl}`"
        )
        r = df_range.iloc[0]
        rows.append({
            "表": tbl,
            "最早日期": r["min_d"] or "—",
            "最新日期": r["max_d"] or "—",
            "记录数": f"{int(r['cnt']):,}",
            "标的数": int(r["codes"]),
        })
    except Exception as e:
        rows.append({"表": tbl, "最早日期": "—", "最新日期": f"查询失败: {e}", "记录数": 0, "标的数": 0})

display(pd.DataFrame(rows))

,表,最早日期,最新日期,记录数,标的数
0,stock_daily,20210408,20260430,"12,254",10
1,stock_technical,20210510,20260430,"12,064",10
2,stock_fundamental,20210408,20260430,"6,065,976",5705
3,stock_cashflow,20260330,20260430,125,10
4,stock_margin,20210412,20260430,166,3
5,stock_holder_count,—,"查询失败: (pymysql.err.OperationalError) (1054, ""U...",0,0
6,index_daily,20210408,20260430,"6,135",5
7,etf_daily,20210408,20260430,"4,858",4


## 5. 各标的数据完整度明细

stock_pool 中每只标的在各表的数据量与最新日期。

In [26]:
from invest_model.repositories.stock_pool_repo import StockPoolRepository

pool_repo = StockPoolRepository(engine)
pool_df = pool_repo.get_pool("core")
etf_df = pool_repo.get_pool("etf")

detail_tables = ["stock_daily", "stock_technical", "stock_fundamental"]

def _code_detail(code, name, tables):
    row = {"代码": code, "名称": name}
    for tbl in tables:
        try:
            r = base_repo.read_sql(
                f"SELECT COUNT(*) as cnt, MAX(trade_date) as latest "
                f"FROM `{tbl}` WHERE code = :code",
                {"code": code},
            ).iloc[0]
            row[f"{tbl}|行数"] = int(r["cnt"])
            row[f"{tbl}|最新"] = r["latest"] or "—"
        except Exception:
            row[f"{tbl}|行数"] = 0
            row[f"{tbl}|最新"] = "—"
    return row

rows = []
if not pool_df.empty:
    for _, s in pool_df.iterrows():
        rows.append(_code_detail(s["code"], s.get("name", ""), detail_tables))

if not etf_df.empty:
    for _, s in etf_df.iterrows():
        rows.append(_code_detail(s["code"], s.get("name", ""), ["etf_daily"]))

if rows:
    df_detail = pd.DataFrame(rows)
    print(f"共 {len(rows)} 只标的\n")
    display(df_detail)
else:
    print("stock_pool 为空，请先配置标的池（参考 02_stock_pool.ipynb）")

共 6 只标的



,代码,名称,stock_daily|行数,stock_daily|最新,stock_technical|行数,stock_technical|最新,stock_fundamental|行数,stock_fundamental|最新,etf_daily|行数,etf_daily|最新
0,000833.SZ,粤桂股份,1225.0,20260430,1206.0,20260430,1185.0,20260430,NaN,NaN
1,002594.SZ,比亚迪,1225.0,20260430,1206.0,20260430,1182.0,20260430,NaN,NaN
2,002648.SZ,卫星化学,1225.0,20260430,1206.0,20260430,1181.0,20260430,NaN,NaN
3,300442.SZ,润泽科技,1219.0,20260430,1200.0,20260430,1189.0,20260430,NaN,NaN
4,600691.SH,潞化科技,1225.0,20260430,1206.0,20260430,1178.0,20260430,NaN,NaN
5,516120.SH,化工50ETF,NaN,NaN,NaN,NaN,NaN,NaN,1225.0,20260430


## 6. 数据质量校验

对 stock_daily 做深度校验：缺失交易日、价格合理性、涨跌幅异常、低成交量。

In [27]:
from invest_model.validators.data_validator import DataValidator

validator = DataValidator(engine)

stock_codes = pool_repo.get_pool_codes("core")
if stock_codes:
    report = validator.validate_stock_daily(stock_codes)
    df_val = report.to_dataframe()

    passed_cnt = df_val["passed"].sum()
    total_cnt = len(df_val)
    warn_cnt = total_cnt - passed_cnt
    print(f"校验结果: {total_cnt} 项检查, {passed_cnt} 通过, {warn_cnt} 警告\n")

    failed = df_val[~df_val["passed"]]
    if failed.empty:
        print("✅ 所有校验项均通过")
    else:
        print(f"⚠️ 以下 {len(failed)} 项存在问题:")
        display(failed)

    print(f"\n完整校验明细:")
    display(df_val)
else:
    print("无个股标的，跳过校验")

校验结果: 20 项检查, 15 通过, 5 警告

⚠️ 以下 5 项存在问题:


,code,check,passed,message
0,000833.SZ,缺失交易日,False,"缺失 64 天 (前5: ['20210104', '20210105', '2021010..."
4,002594.SZ,缺失交易日,False,"缺失 64 天 (前5: ['20210104', '20210105', '2021010..."
8,002648.SZ,缺失交易日,False,"缺失 64 天 (前5: ['20210104', '20210105', '2021010..."
12,300442.SZ,缺失交易日,False,"缺失 70 天 (前5: ['20210104', '20210105', '2021010..."
16,600691.SH,缺失交易日,False,"缺失 64 天 (前5: ['20210104', '20210105', '2021010..."



完整校验明细:


,code,check,passed,message
0,000833.SZ,缺失交易日,False,"缺失 64 天 (前5: ['20210104', '20210105', '2021010..."
1,000833.SZ,价格合理性,True,"零/负价: 0, 范围异常: 0"
2,000833.SZ,涨跌幅异常,True,超过 ±22.0% 的记录: 0 条
3,000833.SZ,低成交量,True,成交量 < 100 手的记录: 0 条
4,002594.SZ,缺失交易日,False,"缺失 64 天 (前5: ['20210104', '20210105', '2021010..."
5,002594.SZ,价格合理性,True,"零/负价: 0, 范围异常: 0"
6,002594.SZ,涨跌幅异常,True,超过 ±22.0% 的记录: 0 条
7,002594.SZ,低成交量,True,成交量 < 100 手的记录: 0 条
8,002648.SZ,缺失交易日,False,"缺失 64 天 (前5: ['20210104', '20210105', '2021010..."
9,002648.SZ,价格合理性,True,"零/负价: 0, 范围异常: 0"


## 7. 总结

In [28]:
from datetime import datetime

elapsed = _time.time() - _t0

latest_date = base_repo.read_sql(
    "SELECT MAX(trade_date) as d FROM stock_daily"
).iloc[0]["d"] or "无数据"

step_ok = sum(1 for s in results.values() if s.startswith("OK"))
step_fail = sum(1 for s in results.values() if s.startswith("FAILED"))

print("=" * 50)
print(f"  运行时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  总耗时:   {elapsed:.1f} 秒")
print(f"  管线步骤: {step_ok} 成功 / {step_fail} 失败")
print(f"  数据新鲜度: stock_daily 最新日期 {latest_date}")
if stock_codes:
    print(f"  数据校验: {passed_cnt}/{total_cnt} 通过 ({passed_cnt/total_cnt*100:.0f}%)")
print(f"  标的数量: 个股 {len(pool_df)} 只, ETF {len(etf_df)} 只")
print("=" * 50)
print("\n运行完毕。如需重新采集，直接 Restart & Run All 即可。")

  运行时间: 2026-05-02 18:37:39
  总耗时:   52.5 秒
  管线步骤: 13 成功 / 0 失败
  数据新鲜度: stock_daily 最新日期 20260430
  数据校验: 15/20 通过 (75%)
  标的数量: 个股 5 只, ETF 1 只

运行完毕。如需重新采集，直接 Restart & Run All 即可。
